# Lab 2b — Import a pre-trained model (skip training)

Use this notebook **instead of notebook 2** when no GPU is available for training, or when there is no time to
wait for a training job. It takes the output of a previous training run and registers it exactly as
notebook 2 does, so notebooks 3 (evaluation) and 4 (deployment) run unchanged.

What you need:

* notebook 1 has run in this account, so the validation data notebook 3 evaluates on is in S3;
* `PRETRAINED_S3_URI`: the S3 prefix of a previous run's model output, holding `hf_merged/` (the merged
  OpenVLA checkpoint, ~15 GB) and usually `adapter/`. For a workshop it's provided by the organizers; you can
  also point it at one of your own finished runs (`.../training-output/<job>/output/model/`).

If the prefix is outside this account's default bucket, the notebook copies it in first (server-side,
typically a few minutes for 15 GB). A prefix already in your bucket is registered in place.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE}, capacity: {config.CAPACITY} -> {', '.join(config.TRAIN['instance_types'])}")
print(f"s3 root:  {s3_root}")

In [ ]:
import time

from tqdm.auto import tqdm

from artifacts import copy_prefix, register_model

PRETRAINED_S3_URI = config.PRETRAINED_S3_URI   # e.g. "s3://<bucket>/openvla-lora/training-output/<job>/output/model/"
# Which recipe produced the artifacts. Notebook 3's approval gate is lenient for "smoke", strict otherwise.
PRETRAINED_RUN_MODE = "full"

if not PRETRAINED_S3_URI:
    raise ValueError("Set PRETRAINED_S3_URI (here or in config.py) to the S3 prefix of a previous run's model "
                     "output, i.e. the folder that contains hf_merged/.")
source = PRETRAINED_S3_URI.rstrip("/") + "/"
print(f"source: {source}")

## Copy into your bucket (if needed) and register

In [ ]:
if source.startswith(s3_root.rstrip("/") + "/"):
    model_prefix = source
    print("already in this account's bucket: registering in place")
else:
    model_prefix = f"{s3_root}/pretrained/{time.strftime('%Y%m%d-%H%M%S')}/"
    n = copy_prefix(source, model_prefix, region=region, progress=tqdm)
    print(f"copied {n} objects to {model_prefix}")

merged_s3_uri = model_prefix + "hf_merged/"
package = register_model(merged_s3_uri, source=source, run_mode=PRETRAINED_RUN_MODE, region=region)
print(f"model package: {package['ModelPackageArn']}")

Continue with **3-evaluation.ipynb**.